In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

code_dir = Path.cwd()
if (code_dir / "final_code" / "cd_aware_ad").is_dir():
    code_dir /= "final_code/cd_aware_ad"
elif (code_dir / "cd_aware_ad").is_dir():
    code_dir /= "cd_aware_ad"
sys.path.insert(0, str(code_dir))

from concept_drift_anomaly_detection import point_adjust_f1

RESULT_DIR = code_dir.resolve().parent / "results" / "post_processing"
ACTUAL_COLS = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z", "mag_x", "mag_y", "mag_z"]
PRED_COLS = [f"{column}_pred" for column in ACTUAL_COLS]

##### Set Best Threshold & Evaluation

In [ ]:
def prediction_visualization(df, columns, title,
                        label_col=None, pred_col=None,
                        color_map=None, save=False, outdir="figure"):
    ts = pd.to_datetime(df['timestamp'], errors='coerce')
    feats = [c for c in columns if c != 'timestamp']
    colors = color_map or plt.cm.tab10.colors

    if save:
        os.makedirs(outdir, exist_ok=True)

    fig, axes = plt.subplots(2, 1, figsize=(20, 8), sharex=True)
    ax1, ax2 = axes

    # --- 연속 1 구간 추출 함수 ---
    def spans_from_binary(s):
        v = s.fillna(0).astype(int).values
        idx = np.flatnonzero(np.diff(np.r_[0, v, 0]) != 0)
        return [(idx[i], idx[i+1]-1) for i in range(0, len(idx), 2)]

    # --- 실제값 + label ---
    for i, f in enumerate(feats):
        ax1.plot(ts, df[f], lw=0.8, label=f, color=colors[i % len(colors)])
    if label_col and label_col in df.columns:
        for s, e in spans_from_binary(df[label_col]):
            ax1.axvspan(ts.iloc[s], ts.iloc[e], color='grey', alpha=0.15, ec='green', lw=1)
    ax1.set_title(f"{title} - Actual + Label")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    # --- 예측값 + pred label ---
    for i, f in enumerate(feats):
        p = f"{f}_pred"
        if p in df.columns:
            ax2.plot(ts, df[p], lw=0.8, label=p, color=colors[i % len(colors)])
    if pred_col and pred_col in df.columns:
        for s, e in spans_from_binary(df[pred_col]):
            ax2.axvspan(ts.iloc[s], ts.iloc[e], color='red', ec='red', lw=1)
    ax2.set_title(f"{title} - Prediction + Pred Label")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    # --- 두 그래프의 y축 스케일 통일 ---
    ymin = min(ax1.get_ylim()[0], ax2.get_ylim()[0])
    ymax = max(ax1.get_ylim()[1], ax2.get_ylim()[1])
    ax1.set_ylim(ymin, ymax)
    ax2.set_ylim(ymin, ymax)

    plt.tight_layout()
    if save:
        plt.savefig(os.path.join(outdir, f"{title}.png"), dpi=150)
        plt.close(fig)
    else:
        plt.show()


In [ ]:
def load_forecast(model, learning, abnormal, div, input_context, output_horizon):
    path = (
        RESULT_DIR / "df" / learning / model / f"{abnormal}_all_scenario"
        / f"3st_output_ratio_{div}_{input_context}_{output_horizon}.csv"
    )
    df = pd.read_csv(path)
    df["error_score"] = np.sqrt(
        ((df[ACTUAL_COLS].values - df[PRED_COLS].values) ** 2).mean(axis=1)
    )
    return df


def threshold_predictions(scores, tau_low, tau_high):
    return ((scores < tau_low) | (scores > tau_high)).astype(int)


def smooth_predictions(predictions, window_size, threshold):
    smoothed = np.zeros_like(predictions)
    for i in range(len(predictions) - window_size + 1):
        if predictions[i:i + window_size].mean() >= threshold:
            smoothed[i:i + window_size] = 1
    return smoothed


def score_predictions(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn + 1e-12)
    tnr = tn / (tn + fp + 1e-12)
    return {
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "PA_F1": point_adjust_f1(y_true, y_pred),
        "ACC": accuracy_score(y_true, y_pred),
        "BACC": 0.5 * (tpr + tnr),
        "PREC": precision_score(y_true, y_pred, zero_division=0),
        "REC": recall_score(y_true, y_pred, zero_division=0),
        "FPR": 1.0 - tnr,
    }

In [ ]:
def set_threshold(model, learning="zsl"):
    div, input_context = 8, 512
    output_horizon = input_context // div
    rows = []

    for abnormal in ("instop", "grabbing", "hitting"):
        df = load_forecast(model, learning, abnormal, div, input_context, output_horizon)
        scores = df["error_score"].values
        y_true = df["label"].astype(int).values
        seq_len = len(scores) // 3
        scenes = (scores[:seq_len], scores[seq_len:2 * seq_len], scores[2 * seq_len:])
        medians, mads = [], []
        for scene, acf in zip(scenes, (1884, 2769, 1137)):
            normal = scene[:acf * 7]
            median = np.median(normal)
            medians.append(median)
            mads.append(np.median(np.abs(normal - median)))

        med = sum(medians) / len(medians)
        robust_sigma = 1.4826 * (sum(mads) / len(mads))
        smoothing = [(None, None)] if abnormal == "hitting" else [
            (2, threshold) for threshold in np.arange(0.1, 0.5, 0.25)
        ]
        best_rank = (-1.0, -1.0, -1.0)
        best = None
        for k_low in np.arange(1.2, 3.0, 0.25):
            tau_low = med - k_low * robust_sigma
            for k_high in np.arange(3.8, 15.0, 0.25):
                tau_high = med + k_high * robust_sigma
                base = threshold_predictions(scores, tau_low, tau_high)
                for ratio, t_ratio in smoothing:
                    pred = base
                    if ratio is not None:
                        pred = smooth_predictions(base, output_horizon * ratio, t_ratio)
                    metrics = score_predictions(y_true, pred)
                    rank = (metrics["F1"], metrics["BACC"], metrics["PREC"])
                    if rank > best_rank:
                        best_rank = rank
                        best = {
                            "abnormal": abnormal, "div": div,
                            "input_context": input_context, "output_horizon": output_horizon,
                            "k_low": k_low, "k_high": k_high, **metrics,
                            "tau_low": tau_low, "tau_high": tau_high,
                            "best_ratio": ratio, "best_t_ratio": t_ratio,
                        }
        rows.append(best)
        print(f"{model} | {abnormal} | F1={best['F1']:.4f} | PA-F1={best['PA_F1']:.4f}")

    best_df = pd.DataFrame(rows)
    summary_dir = RESULT_DIR / "summary" / f"fm_{learning}" / model
    summary_dir.mkdir(parents=True, exist_ok=True)
    best_df.to_csv(summary_dir / "3st_all_scenario_div_sweep_all.csv", index=False)
    best_df.to_csv(summary_dir / "3st_all_scenario_div_best_per_pair.csv", index=False)
    return best_df

In [ ]:
def prediction_evaluation(model, best_df=None, learning="zsl", save=False):
    if best_df is None:
        best_df = pd.read_csv(
            RESULT_DIR / "summary" / f"fm_{learning}" / model
            / "3st_all_scenario_div_best_per_pair.csv"
        )

    for row in best_df.itertuples(index=False):
        abnormal = row.abnormal
        div, context, horizon = int(row.div), int(row.input_context), int(row.output_horizon)
        df = load_forecast(model, learning, abnormal, div, context, horizon)
        pred = threshold_predictions(df["error_score"].values, float(row.tau_low), float(row.tau_high))
        print(f"[{abnormal} | div={div}] pred sum = {int(pred.sum())}")

        if abnormal != "hitting" and not (pd.isna(row.best_ratio) or pd.isna(row.best_t_ratio)):
            pred = smooth_predictions(pred, int(horizon * float(row.best_ratio)), float(row.best_t_ratio))
        df["pred_anomaly"] = pred

        prediction_visualization(
            df, ["timestamp", *ACTUAL_COLS],
            title=f"{model} | All Scenario {abnormal}",
            label_col="label", pred_col="pred_anomaly", save=save,
            outdir=RESULT_DIR / "figure" / "foundation_model" / "3st" / learning / model,
        )

In [ ]:
experiments = {
    "zsl": ["chronos2", "timesfm", "chronos-t5-large", "moment", "lagllama"],
    "headonly_finetuned": ["chronos2", "moment", "timesfm"],
}

for learning, models in experiments.items():
    for model in models:
        try:
            if learning == "headonly_finetuned":
                summary = (
                    RESULT_DIR / "summary" / f"fm_{learning}" / model
                    / "3st_finetuned_all_scenario_div_best_per_pair.csv"
                )
                prediction_evaluation(model=model, best_df=pd.read_csv(summary), learning=learning)
            else:
                set_threshold(model=model, learning=learning)
                prediction_evaluation(model=model, learning=learning)
        except FileNotFoundError as error:
            print(f"[SKIP] {learning} | {model} | {error.filename}")